# Section 4 — Gaussian Processes

Gaussian Processes (GPs) are **nonparametric Bayesian models** for functions.  
They place a prior directly over functions $f(\cdot)$, rather than over finite-dimensional parameters.

---

## 4.1 GP Basics

A **Gaussian Process** is a collection of random variables $\{f(x) : x \in \mathcal{X}\}$ such that  
for any finite set $x_1, \dots, x_n$, the vector
$$
\mathbf{f} = [f(x_1), \dots, f(x_n)]^\top
$$
has a multivariate Gaussian distribution:
$$
\mathbf{f} \sim \mathcal{N}\big(\mathbf{m}, \mathbf{K}\big),
$$
where:
- Mean function: $m(x) = \mathbb{E}[f(x)]$
- Covariance function (kernel): $k(x,x') = \mathrm{Cov}(f(x), f(x'))$

We write:
$$
f(x) \sim \mathcal{GP}\big(m(x), k(x,x')\big).
$$

---

## 4.2 Covariance Functions & Kernels

The kernel encodes **smoothness, periodicity, amplitude** of the function.  
It must be **positive semidefinite**.

**Examples:**
- **Squared Exponential (RBF)**:
$$
k_{\mathrm{RBF}}(x,x') = \sigma_f^2 \exp\!\left(-\frac{\|x-x'\|^2}{2\ell^2}\right)
$$
- **Matérn** (e.g., $\nu=3/2$):
$$
k_{\nu=3/2}(x,x') = \sigma_f^2 \left(1 + \frac{\sqrt{3}r}{\ell}\right)\exp\!\left(-\frac{\sqrt{3}r}{\ell}\right),
\quad r = \|x-x'\|.
$$
- **Periodic**:
$$
k_{\mathrm{per}}(x,x') = \sigma_f^2 \exp\!\left( -\frac{2\sin^2(\pi\|x-x'\|/p)}{\ell^2} \right)
$$
- **Linear**:
$$
k_{\mathrm{lin}}(x,x') = \sigma_b^2 + \sigma_v^2 (x - c)(x' - c)
$$

**Properties:**
- **Stationary** kernels: depend only on $r = x - x'$
- Kernels can be **summed** or **multiplied** to create new kernels.

---

## 4.3 GP Regression

**Model:**
$$
y_i = f(x_i) + \varepsilon_i, \quad \varepsilon_i \sim \mathcal{N}(0, \sigma_n^2)
$$
Prior: $f(\cdot) \sim \mathcal{GP}(m(\cdot), k(\cdot,\cdot))$

Let:
- $\mathbf{X} = [x_1,\dots,x_n]^\top$
- $\mathbf{y} = [y_1,\dots,y_n]^\top$
- $\mathbf{K} = [k(x_i,x_j)]_{ij}$
- $\mathbf{K}_y = \mathbf{K} + \sigma_n^2 \mathbf{I}$

### Joint distribution of training and test points
For test inputs $\mathbf{X}^\star$:
$$
\begin{bmatrix}
\mathbf{y} \\ \mathbf{f}^\star
\end{bmatrix}
\sim \mathcal{N}\!\left(
\begin{bmatrix}
\mathbf{m} \\ \mathbf{m}^\star
\end{bmatrix},
\begin{bmatrix}
\mathbf{K}_y & \mathbf{K}_\star^\top \\
\mathbf{K}_\star & \mathbf{K}_{\star\star}
\end{bmatrix}
\right)
$$
where:
- $\mathbf{K}_\star = [k(x_i, x_j^\star)]_{ij}$
- $\mathbf{K}_{\star\star} = [k(x_i^\star, x_j^\star)]_{ij}$

### Posterior predictive
$$
\mathbf{f}^\star \mid \mathbf{X},\mathbf{y},\mathbf{X}^\star \sim \mathcal{N}\!\left(
\bar{\mathbf{f}}^\star,\; \mathrm{Cov}(\mathbf{f}^\star)
\right)
$$
with:
$$
\bar{\mathbf{f}}^\star = \mathbf{m}^\star + \mathbf{K}_\star^\top \mathbf{K}_y^{-1}(\mathbf{y} - \mathbf{m})
$$
$$
\mathrm{Cov}(\mathbf{f}^\star) = \mathbf{K}_{\star\star} - \mathbf{K}_\star^\top \mathbf{K}_y^{-1} \mathbf{K}_\star
$$

---

## 4.4 GP Classification

- **Likelihood**: Non-Gaussian (e.g., Bernoulli with probit or logistic link)
- Posterior over $f$ is **not Gaussian** $\Rightarrow$ need approximation:
  - **Laplace approximation**: Gaussian approx around MAP
  - **EP** (Expectation Propagation)
  - **Variational inference**

**Predictive probability**:
$$
p(y^\star=1 \mid \mathbf{X},\mathbf{y},x^\star) = \int \sigma(f^\star) \, p(f^\star \mid \mathbf{X},\mathbf{y},x^\star) \, df^\star
$$
where $\sigma(\cdot)$ is sigmoid/probit.

---

## 4.5 Model Evidence / Marginal Likelihood

For GP regression:
$$
p(\mathbf{y} \mid \mathbf{X}, \theta) =
\frac{1}{(2\pi)^{n/2} |\mathbf{K}_y|^{1/2}}
\exp\!\left( -\frac12 (\mathbf{y} - \mathbf{m})^\top \mathbf{K}_y^{-1} (\mathbf{y} - \mathbf{m}) \right)
$$
where $\theta$ are kernel hyperparameters (lengthscale, variance, noise).

Log marginal likelihood:
$$
\log p(\mathbf{y}) = -\frac12 (\mathbf{y} - \mathbf{m})^\top \mathbf{K}_y^{-1} (\mathbf{y} - \mathbf{m})
 - \frac12 \log |\mathbf{K}_y| - \frac{n}{2} \log 2\pi
$$

---

## 4.6 Hyperparameter Optimization in GPs

- Maximize $\log p(\mathbf{y} \mid \mathbf{X}, \theta)$ wrt $\theta$ (type-II MLE / empirical Bayes)
- Use gradients:
$$
\frac{\partial \log p(\mathbf{y})}{\partial \theta_j} = \frac12 \mathbf{y}^\top \mathbf{K}_y^{-1} \frac{\partial \mathbf{K}_y}{\partial \theta_j} \mathbf{K}_y^{-1} \mathbf{y} - \frac12 \mathrm{tr}\!\left(\mathbf{K}_y^{-1} \frac{\partial \mathbf{K}_y}{\partial \theta_j}\right)
$$
- Can place **hyperpriors** on $\theta$ and infer them via MCMC/VI.

---

## 4.7 Exam Tips

- Always write the GP prior **fully**: $f(x) \sim \mathcal{GP}(m(x), k(x,x'))$.  
- For regression: write **joint Gaussian** of train + test, then **condition**.  
- Include **noise variance** in $\mathbf{K}_y$ for observed $y$.  
- For classification: state the **likelihood**, note **non-Gaussian posterior**, and the approximation method.  
- Remember **log marginal likelihood** form and the $O(n^3)$ complexity (due to Cholesky of $\mathbf{K}_y$).

---